# Paper-to-Code 3 – LightGCN (2020)
**Paper:** He, Deng, Wang, Li, Zhang, Wang – *LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation* – [arXiv:2002.02126](https://arxiv.org/abs/2002.02126) · builds on NGCF ([arXiv:1905.08108](https://arxiv.org/abs/1905.08108)) · notes: [[Graph Neural Networks]], [[Collaborative Filtering and Matrix Factorization]] · plan: [[Paper-to-Code Months]]

### The claim we reproduce
> For collaborative filtering, a GCN needs **neither feature transformations nor nonlinearities**. Just smooth the ID embeddings over the user–item graph, $E^{(k+1)} = \tilde A E^{(k)}$ with $\tilde A = D^{-1/2} A D^{-1/2}$, and average the layers. This beats plain matrix factorisation (and NGCF).

### Our scaled-down setup
We reuse the **exact split and evaluation** from [[Project - MovieLens Recommender]] (MovieLens *latest-small*, positives = rating ≥ 4, leave-last-out, HR@10 / NDCG@10 against 99 sampled negatives), so you can compare with your M2/M3 notebooks. Note that **$K=0$ layers is exactly BPR-MF**, so the baseline comes for free. 60 epochs ≈ 1 min on a CPU. The paper used Gowalla / Yelp2018 / Amazon-Book with full-ranking Recall@20.

**How this works:** cells marked **TODO** are yours (each one is a small function). Run the ✅ check under it.
Experiment cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*.
Everything runs on a laptop CPU – the paper's full-scale setup is listed under *Scale it up*.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, pathlib, urllib.request, zipfile, io, time, warnings
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); torch.set_num_threads(4); warnings.filterwarnings("ignore", message="Sparse invariant")
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the experiment cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

DATA = pathlib.Path("data/ml-latest-small")
if not DATA.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall("data")
ratings = pd.read_csv(DATA / "ratings.csv"); movies = pd.read_csv(DATA / "movies.csv")

# Same split as the MovieLens project (M1–M5): leave-last-out by time, positives = rating >= 4
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(movies.movieId.unique()))}
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HR@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (same as the project notebooks)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum() + rng.integers(0, (s == s[0]).sum())
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

u_tr = torch.tensor(train.userId.map(uid).to_numpy()); i_tr = torch.tensor(train.movieId.map(iid).to_numpy())

## 1. The normalised adjacency matrix (paper Eq. 7–8)
Put users and items in **one** graph with $N = n_\text{users} + n_\text{items}$ nodes: user $u$ is node $u$, item $i$ is node $n_\text{users} + i$. Every training interaction is an undirected edge. Return the sparse $N\times N$ matrix
$$\tilde A_{ab} = \frac{1}{\sqrt{d_a}\sqrt{d_b}} \text{ for every edge } (a,b)$$
Hint: build both directions of each edge, `deg = torch.bincount(rows, minlength=N)`, then `torch.sparse_coo_tensor(indices, values, (N, N)).coalesce()`.

In [ ]:
def normalized_adjacency(u, i, n_users, n_items):
    # TODO 1: symmetric D^-1/2 A D^-1/2 of the user–item bipartite graph, as a sparse (N, N) tensor
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_adj():   # users 0,1 – items 0,1 – edges u0-i0, u0-i1, u1-i0  →  degrees u0=2, u1=1, i0=2, i1=1
    A = normalized_adjacency(torch.tensor([0, 0, 1]), torch.tensor([0, 1, 0]), 2, 2).to_dense()
    want = torch.zeros(4, 4); r2 = 2 ** -0.5
    want[0, 2] = want[2, 0] = 0.5; want[0, 3] = want[3, 0] = r2; want[1, 2] = want[2, 1] = r2
    return torch.allclose(A, want)
check("adjacency", _t_adj, "On the 2-user/2-item toy graph expect A[u0,i0] = 1/√(2·2) = 0.5 and A[u0,i1] = A[u1,i0] = 1/√2, symmetric, zeros elsewhere.")

## 2. Light graph convolution + layer combination (paper Eq. 3–4)
$E^{(0)}$ is the only trainable parameter. Propagate $E^{(k+1)} = \tilde A E^{(k)}$ for $k = 0..K-1$ – **no weight matrix, no activation** – and return the mean $\frac{1}{K+1}\sum_{k=0}^{K} E^{(k)}$ (the paper's $\alpha_k = 1/(K+1)$).

In [ ]:
class LightGCN(nn.Module):
    def __init__(self, A, n_nodes, K=3, dim=64):
        super().__init__()
        self.A, self.K = A, K
        self.emb = nn.Embedding(n_nodes, dim); nn.init.normal_(self.emb.weight, std=0.1)   # as in the official code

    def forward(self):
        # TODO 2: final embeddings of all N nodes = mean over layers 0..K of A^k E0
        raise NotImplementedError  # TODO 2

In [ ]:
def _t_prop():
    A = torch.tensor([[0., 1.], [1., 0.]]).to_sparse()
    m0, m2 = LightGCN(A, 2, K=0, dim=3), LightGCN(A, 2, K=2, dim=3)
    m2.emb.weight.data = m0.emb.weight.data.clone(); E = m0.emb.weight
    swapped = E.flip(0)                                    # A·E swaps the two rows; A²·E = E
    return torch.allclose(m0(), E) and torch.allclose(m2(), (E + swapped + E) / 3)
check("propagation", _t_prop, "K=0 must return E0 itself; K=2 must return (E0 + A·E0 + A²·E0) / 3.")

## 3. BPR loss (paper Eq. 15)
$\mathcal{L} = -\frac1B\sum \ln\sigma(e_u^\top e_i - e_u^\top e_j)$ for (user, positive item, sampled negative item). (L2 on $E^{(0)}$ is added in the training loop.)

In [ ]:
def bpr_loss(e_u, e_i, e_j):
    # TODO 3: mean of -log sigmoid(score_pos - score_neg)
    raise NotImplementedError  # TODO 3

In [ ]:
check("BPR loss", lambda: abs(bpr_loss(*(torch.zeros(5, 4),) * 3).item() - np.log(2)) < 1e-6
      and bpr_loss(torch.ones(1, 4), torch.ones(1, 4), -torch.ones(1, 4)).item() < 1e-3,
      "Equal scores must give log 2 = 0.693; a much higher positive score must give ≈ 0.")

## 4. Experiment – MF ($K=0$) vs LightGCN ($K=3$)
Identical everything (Adam lr 0.005, batch 2048, 64-d, L2 1e-4, 60 epochs, seed 0) – only the number of propagation layers changes.

In [ ]:
EPOCHS, LR, BATCH, L2 = 60, 0.005, 2048, 1e-4
results, curves = {}, {}

if ready("adjacency", "propagation", "BPR loss"):
    A = normalized_adjacency(u_tr, i_tr, n_users, n_items)
    seen = torch.zeros(n_users, n_items, dtype=torch.bool); seen[u_tr, i_tr] = True

    def sample_negatives(u):
        j = torch.randint(0, n_items, (len(u),)); bad = seen[u, j]
        while bad.any():
            j[bad] = torch.randint(0, n_items, (int(bad.sum()),)); bad = seen[u, j]
        return j

    def scorer(model):
        with torch.no_grad(): E = model()
        U, I = E[:n_users], E[n_users:]
        return lambda u, cand: (U[uid[u]] @ I[torch.tensor([iid[x] for x in cand])].T).numpy()

    for K in (0, 3):
        torch.manual_seed(0)
        model = LightGCN(A, n_users + n_items, K=K); opt = torch.optim.Adam(model.parameters(), lr=LR)
        t, curves[K] = time.time(), []
        for ep in range(EPOCHS):
            perm = torch.randperm(len(u_tr))
            for b in range(0, len(perm), BATCH):
                idx = perm[b:b + BATCH]; u, i = u_tr[idx], i_tr[idx]; j = sample_negatives(u)
                E = model(); E0 = model.emb.weight
                loss = bpr_loss(E[u], E[n_users + i], E[n_users + j])
                reg = L2 * (E0[u].pow(2).sum() + E0[n_users + i].pow(2).sum() + E0[n_users + j].pow(2).sum()) / (2 * len(u))
                opt.zero_grad(); (loss + reg).backward(); opt.step()
            if ep % 10 == 9:
                curves[K].append(evaluate(scorer(model), val, train)["NDCG@k"])
        results[K] = evaluate(scorer(model), test, train)
        print(f"K={K} ({'MF-BPR' if K == 0 else 'LightGCN'}): test {results[K]}  ({time.time()-t:.0f}s)")
    for K, c in curves.items():
        plt.plot(range(10, EPOCHS + 1, 10), c, "o-", label="MF-BPR (K=0)" if K == 0 else f"LightGCN (K={K})")
    plt.xlabel("epoch"); plt.ylabel("validation NDCG@10"); plt.legend(); plt.show()

In [ ]:
if ready("adjacency", "propagation", "BPR loss"):
    check("LightGCN beats MF", lambda: results[3]["NDCG@k"] > results[0]["NDCG@k"] + 0.01,
          "Expected K=3 to beat K=0 by at least 0.01 test NDCG@10 – check the normalisation and the layer mean.")

<details><summary>▶ Solution</summary>

```python
def normalized_adjacency(u, i, n_users, n_items):
    N = n_users + n_items
    rows = torch.cat([u, i + n_users]); cols = torch.cat([i + n_users, u])
    deg = torch.bincount(rows, minlength=N).float()
    vals = deg[rows].pow(-0.5) * deg[cols].pow(-0.5)
    return torch.sparse_coo_tensor(torch.stack([rows, cols]), vals, (N, N)).coalesce()
```

```python
    def forward(self):
        e = self.emb.weight
        layers = [e]
        for _ in range(self.K):
            e = torch.sparse.mm(self.A, e)
            layers.append(e)
        return torch.stack(layers).mean(0)
```

```python
def bpr_loss(e_u, e_i, e_j):
    return -F.logsigmoid((e_u * e_i).sum(-1) - (e_u * e_j).sum(-1)).mean()
```
</details>

## Scale it up (stretch goals)
- **Ablation that *is* the paper:** add NGCF's per-layer weight matrix $W^{(k)}$ and a LeakyReLU (`e = act(A @ e @ W)`) and show it does *not* help (paper Table 1/4).
- Sweep $K = 0..4$ and plot NDCG@10 vs $K$ (paper Fig. 4); try *only the last layer* instead of the mean (over-smoothing!).
- Switch to full-ranking Recall@20 / NDCG@20 like the paper (the sampled metric can reorder models – see [[Evaluating Recommenders]]).
- Run on Gowalla (the paper's data, linked from the official repo) with 1000 epochs and lr 0.001 (GPU).

---
## 📝 Write-up (fill in – half a page is enough)
Copy this into your results log in [[Paper-to-Code Months]] when done.

- **Paper and its claim in one sentence:**
- **What I implemented / what I scaled down (and why):**
- **Result:**

  | setting | metric | paper | me |
  |---|---|---|---|
  |  |  |  |  |

- **Did the claim hold at small scale? Why / why not?**
- **One detail the paper glossed over that I had to figure out:**
- **What I would try next with a GPU:**
